# 01_gold_daily_market_summary

## Purpose

Create the Gold daily market summary table.

This notebook aggregates cleaned market price Silver data into a business-facing Gold output.

## Expected output

`vattenfall_dev.analytics.gold_daily_market_summary`

## Expected grain

One row per report day and region.

## Expected KPIs

- average market price
- minimum market price
- maximum market price
- total market volume
- market record count

## Main idea

This Gold table provides daily regional market context for business users and dashboard reporting.

In [0]:
import sys
import importlib

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

import transforms.gold_aggregations as gold_aggregations
import metrics.business_metrics as business_metrics
from utils.logging_utils import log_table_operation, log_row_count

importlib.reload(gold_aggregations)
importlib.reload(business_metrics)

In [0]:
catalog = "vattenfall_dev"

source_table = f"{catalog}.refined.silver_market_prices"
target_table = f"{catalog}.analytics.gold_daily_market_summary"

print("Gold table:", "Daily Market Summary")
print("Source table:", source_table)
print("Target table:", target_table)
print("Grain: one row per report_day and region")

In [0]:
market_df = spark.table(source_table)

source_count = market_df.count()

log_row_count("Source Silver market rows", source_count)

display(market_df.limit(20))

In [0]:
gold_market_df = (
    gold_aggregations.aggregate_daily_market_summary(market_df)
    .transform(business_metrics.add_market_pressure_flag)
)

gold_count = gold_market_df.count()

log_row_count("Gold daily market summary rows", gold_count)

display(gold_market_df)

In [0]:
(
    gold_market_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

log_table_operation(
    source_table=source_table,
    target_table=target_table,
    operation_name="Create Gold daily market summary"
)

print("Written Gold table:", target_table)

In [0]:
gold_df = spark.table(target_table)

print("Rows in Gold table:", gold_df.count())
print("Columns:", gold_df.columns)

display(gold_df)

In [0]:
print("High market price flag distribution:")

gold_df.groupBy("is_high_market_price").count().show()

print("Null checks for Gold grain:")

for column_name in ["report_day", "region"]:
    null_count = gold_df.filter(f"{column_name} IS NULL").count()
    print(column_name, "nulls:", null_count)

print("Duplicate grain check:")

duplicate_count = (
    gold_df
    .groupBy("report_day", "region")
    .count()
    .filter("count > 1")
    .count()
)

print("duplicate day-region rows:", duplicate_count)